# TP-1 : Survie sur le Titanic — Classification

**Objectif** : Prédire la survie des passagers du Titanic à partir de leurs caractéristiques.

**Dataset** : [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)

**Compétences** :
- Prétraitement des données (valeurs manquantes, encodage)
- Feature Engineering
- Classification avec Random Forest
- Évaluation des modèles

##  Table des matières

1. [Import des bibliothèques](#section-1)
2. [Chargement et exploration des données](#section-2)
3. [Prétraitement des données](#section-3)
4. [Feature Engineering](#section-4)
5. [Modélisation](#section-5)
6. [Évaluation et interprétation](#section-6)

<a id='section-1'></a>
## 1️⃣ Import des bibliothèques

In [ ]:
# Manipulation de données
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Bibliothèques importées avec succès !")

<a id='section-2'></a>
## 2️⃣ Chargement et exploration des données

In [ ]:
# Chargement des données
# Note: Sur Kaggle, utilisez directement le chemin /kaggle/input/
train_df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

print(f"📊 Dimensions du dataset : {train_df.shape}")
print(f"\n Colonnes : {list(train_df.columns)}")

In [ ]:
# Aperçu des données
train_df.head(10)

In [ ]:
# Informations sur les données
train_df.info()

In [ ]:
# Statistiques descriptives
train_df.describe()

In [ ]:
# Valeurs manquantes
missing_values = train_df.isnull().sum()
missing_percent = (missing_values / len(train_df)) * 100

missing_df = pd.DataFrame({
    'Valeurs manquantes': missing_values,
    'Pourcentage': missing_percent
}).sort_values('Pourcentage', ascending=False)

print("Valeurs manquantes :")
print(missing_df[missing_df['Valeurs manquantes'] > 0])

### 📊 Visualisation de la distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Distribution de la survie
sns.countplot(data=train_df, x='Survived', ax=axes[0, 0])
axes[0, 0].set_title('Distribution de la survie')
axes[0, 0].set_xticklabels(['Non survécu', 'Survécu'])

# Survie par sexe
sns.countplot(data=train_df, x='Sex', hue='Survived', ax=axes[0, 1])
axes[0, 1].set_title('Survie par sexe')

# Survie par classe
sns.countplot(data=train_df, x='Pclass', hue='Survived', ax=axes[0, 2])
axes[0, 2].set_title('Survie par classe')

# Distribution de l'âge
sns.histplot(data=train_df, x='Age', hue='Survived', bins=30, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribution de l\'âge par survie')

# Distribution du prix du billet
sns.histplot(data=train_df, x='Fare', hue='Survived', bins=30, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Distribution du prix par survie')

# Survie par port d'embarquement
sns.countplot(data=train_df, x='Embarked', hue='Survived', ax=axes[1, 2])
axes[1, 2].set_title('Survie par port d\'embarquement')

plt.tight_layout()
plt.show()

<a id='section-3'></a>
## 3️⃣ Prétraitement des données

In [ ]:
# Création d'une copie pour le prétraitement
df = train_df.copy()

# Suppression des colonnes inutiles
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

print("🗑️ Colonnes supprimées : PassengerId, Name, Ticket, Cabin")

In [ ]:
# Gestion des valeurs manquantes

# Age : remplissage par la médiane
df['Age'].fillna(df['Age'].median(), inplace=True)

# Embarked : remplissage par le mode (valeur la plus fréquente)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Fare : remplissage par la médiane
df['Fare'].fillna(df['Fare'].median(), inplace=True)

print("✅ Valeurs manquantes traitées")
print(f"Valeurs manquantes restantes : {df.isnull().sum().sum()}")

In [ ]:
# Encodage des variables catégorielles

# Sex : Male=0, Female=1
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Embarked : One-hot encoding
df = pd.get_dummies(df, columns=['Embarked'], prefix='Embarked')

print("✅ Variables catégorielles encodées")
df.head()

<a id='section-4'></a>
## 4️⃣ Feature Engineering

In [ ]:
# Création de nouvelles features

# FamilySize : taille de la famille
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# IsAlone : voyage seul ou non
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# AgeGroup : groupes d'âge
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], 
                        labels=['Enfant', 'Adolescent', 'Adulte', 'Mature', 'Senior'])

# FarePerPerson : prix par personne
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

print("✅ Nouvelles features créées :")
print("  - FamilySize : taille de la famille")
print("  - IsAlone : voyage seul")
print("  - AgeGroup : groupe d'âge")
print("  - FarePerPerson : prix par personne")

In [ ]:
# Encodage de AgeGroup
df = pd.get_dummies(df, columns=['AgeGroup'], prefix='Age')

df.head()

<a id='section-5'></a>
## 5️⃣ Modélisation

In [ ]:
# Séparation des features et de la cible
X = df.drop('Survived', axis=1)
y = df['Survived']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Train set : {X_train.shape[0]} échantillons")
print(f"📊 Test set : {X_test.shape[0]} échantillons")

In [ ]:
# Modèle 1 : Régression Logistique
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_pred)

print(f" Régression Logistique - Accuracy : {lr_accuracy:.4f}")

In [ ]:
# Modèle 2 : Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"🌲 Random Forest - Accuracy : {rf_accuracy:.4f}")

In [ ]:
# Optimisation des hyperparamètres avec GridSearch
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

print(" Optimisation en cours...")
grid_search.fit(X_train, y_train)

print(f"\n✅ Meilleurs paramètres : {grid_search.best_params_}")
print(f"✅ Meilleure accuracy CV : {grid_search.best_score_:.4f}")

<a id='section-6'></a>
## 6️⃣ Évaluation et interprétation

In [ ]:
# Meilleur modèle
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test)

print(" Rapport de classification :")
print(classification_report(y_test, best_pred, target_names=['Non survécu', 'Survécu']))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non survécu', 'Survécu'],
            yticklabels=['Non survécu', 'Survécu'])
plt.title('Matrice de confusion')
plt.ylabel('Vrai label')
plt.xlabel('Prédiction')
plt.show()

In [ ]:
# Importance des features
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10), x='importance', y='feature', palette='viridis')
plt.title('Top 10 - Importance des features')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation finale
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='accuracy')

print(f" Cross-validation (5-fold) :")
print(f"  Mean accuracy : {cv_scores.mean():.4f}")
print(f"  Std accuracy : {cv_scores.std():.4f}")
print(f"  Scores : {cv_scores}")

##  Conclusion

Dans ce TP, nous avons :

1. ✅ **Exploré** le dataset Titanic et identifié les patterns clés
2. ✅ **Prétraité** les données (valeurs manquantes, encodage)
3. ✅ **Créé** de nouvelles features pertinentes
4. ✅ **Entraîné** plusieurs modèles de classification
5. ✅ **Optimisé** les hyperparamètres avec GridSearch
6. ✅ **Évalué** le modèle et analysé l'importance des features

**Résultat** : Accuracy de ~82% avec Random Forest optimisé.

**Prochaines étapes** :
- Tester d'autres algorithmes (XGBoost, SVM)
- Explorer plus de features (extraction du titre du nom)
- Soumettre sur Kaggle pour voir le score public